### IBM QAOA Parameter Setting Analysis ###
This notebook produces tables and generates plots to analyze real IBM Qunatum hardware data while running QAOA on ensembles of the MaxCut problem. Various training strategies are utilized to determine the optimal parameters (angles of QAOA) before running them on hardware. The idea is to create a resource-cost estimation for these different parameter setting strategies and suggest best practices.

In [1]:
%load_ext autoreload
%autoreload 2

# Path imports
import sys
import os
sys.path.insert(0, os.path.abspath('..'))

# IBM QAOA package specific
from src.Processing import QAOAHardware, QAOATraining
from src.Processing import set_data_path
from src.Processing import load_problem_instance
from src.Appox_Ratio_Calc import get_minmax, extract_minmax_args, maxcut_approximation_ratio

# General useful Python libraries
import pandas as pd 
import matplotlib.pyplot as plt 
import numpy as np  
from pathlib import Path

# Add stochastic-benchmark src to path and import related libraries
sys.path.append('../../src')
import stochastic_benchmark as SB
import bootstrap
import interpolate
import stats
from utils_ws import *
from plotting import Plotting

#### Data Analysis for IBM Quantum Hardware Data ####

In [2]:
# Select graph to explore
graph_type = "heavy_hex"

# Set data path
data_dir = "/mnt/c/Users/rames102/Desktop/QAOA-Parameter-Setting/data"
# Set instance path
inst_path = "/mnt/c/Users/rames102/Desktop/QAOA-Parameter-Setting/instances"
# Set minmax path for approximation ratio calculations later on (specific to evaluations pertaining the Maccut problem)
minmax_path = "/mnt/c/Users/rames102/Desktop/QAOA-Parameter-Setting/data/minmax_cuts"

# Set hardware data path to graph type
hardware_data_path = set_data_path(data_dir, True, False, graph_type)
print(hardware_data_path)

# Set training data path to graph type
training_data_path = set_data_path(data_dir, False, True, graph_type)
print(training_data_path)

# # Load instance path
# instance_path = load_problem_instance("heavy_hex")
# print(instance_path)

# Select hardware instance parameters to explore:
p_list = [5, 10] # eg. QAOA depths for heavy_hex
num_nodes = 144 # eg. Problem size
instance_list = [0, 1, 2, 3, 4, 5, 6, 7, 8, 9] # eg. instance index (last index only)

# Load heavy_hex instances for the above parameter combination
instance_paths_hardware = []

for p in p_list:
    for instance in instance_list:
        instance_paths_hardware.extend(QAOAHardware.locate_hardware_instance(hardware_data_path, 
        graph_type, str(instance), str(num_nodes), str(p))) # extend ensures flat list
print(instance_paths_hardware)
print(len(instance_paths_hardware)) 

/mnt/c/Users/rames102/Desktop/QAOA-Parameter-Setting/data/hardware/heavy_hex
/mnt/c/Users/rames102/Desktop/QAOA-Parameter-Setting/data/training/heavy_hex
[PosixPath('/mnt/c/Users/rames102/Desktop/QAOA-Parameter-Setting/data/hardware/heavy_hex/000N144HH73_5_d4am1h7nmdfs73ad2p90.json'), PosixPath('/mnt/c/Users/rames102/Desktop/QAOA-Parameter-Setting/data/hardware/heavy_hex/001N144HH73_5_d4mkjm90i6jc73dgt05g.json'), PosixPath('/mnt/c/Users/rames102/Desktop/QAOA-Parameter-Setting/data/hardware/heavy_hex/002N144HH73_5_d4mkjrt74pkc738940t0.json'), PosixPath('/mnt/c/Users/rames102/Desktop/QAOA-Parameter-Setting/data/hardware/heavy_hex/003N144HH73_5_d4mkk0p0i6jc73dgt0fg.json'), PosixPath('/mnt/c/Users/rames102/Desktop/QAOA-Parameter-Setting/data/hardware/heavy_hex/004N144HH73_5_d4mkk610i6jc73dgt0kg.json'), PosixPath('/mnt/c/Users/rames102/Desktop/QAOA-Parameter-Setting/data/hardware/heavy_hex/005N144HH73_5_d4mkkbav0j9c73e6e1dg.json'), PosixPath('/mnt/c/Users/rames102/Desktop/QAOA-Parameter-Set

In [3]:
all_instances_data_hardware = []

for instance_path in instance_paths_hardware:
    all_instances_data_hardware.extend(QAOAHardware.load_hardware_instance(instance_path))

print(all_instances_data_hardware)

# Expand each object into data columns
all_data_dicts_hardware = []

for object in all_instances_data_hardware:
    data_dict_hardware = {

        "instance_name" : object.instance_name,
        "QPU_time (s)" : object.QPU_time,
        "num_shots" : object.num_shots,
        "problem_class" : object.problem_class,
        "training_method" : object.training_method,
        "expected_energy" : object.expected_energy,
        "result_file" : object.result_file
    }
    all_data_dicts_hardware.append(data_dict_hardware)

[<src.Processing.QAOAHardware object at 0x71c20b01f460>, <src.Processing.QAOAHardware object at 0x71c20b01dc60>, <src.Processing.QAOAHardware object at 0x71c20b01e020>, <src.Processing.QAOAHardware object at 0x71c20b01d990>, <src.Processing.QAOAHardware object at 0x71c20b01d3c0>, <src.Processing.QAOAHardware object at 0x71c20b01dfc0>, <src.Processing.QAOAHardware object at 0x71c20b01f7c0>, <src.Processing.QAOAHardware object at 0x71c20b01d600>, <src.Processing.QAOAHardware object at 0x71c20b01d9c0>, <src.Processing.QAOAHardware object at 0x71c20b01f4f0>, <src.Processing.QAOAHardware object at 0x71c20b01ffd0>, <src.Processing.QAOAHardware object at 0x71c20b01c550>, <src.Processing.QAOAHardware object at 0x71c20b01d630>, <src.Processing.QAOAHardware object at 0x71c20b01c130>, <src.Processing.QAOAHardware object at 0x71c20b01c610>, <src.Processing.QAOAHardware object at 0x71c20b01e2f0>, <src.Processing.QAOAHardware object at 0x71c20b01f790>, <src.Processing.QAOAHardware object at 0x71c20b

In [4]:
# Creata a DataFrame to document the extracted hardware data  
df_hardware = pd.DataFrame(all_data_dicts_hardware)
print(df_hardware.head())

  instance_name  QPU_time (s)  num_shots problem_class training_method  \
0   000N144HH73             8       4096        maxcut        F_MPS_10   
1   000N144HH73             8       4096        maxcut         F_PP_10   
2   000N144HH73             8       4096        maxcut        I_MPS_10   
3   000N144HH73             8       4096        maxcut         I_PP_10   
4   000N144HH73             8       4096        maxcut    TQA_PP_opt_5   

   expected_energy                           result_file  
0        27.752374  000N144HH73_MC_F_MPS_optBD24_10.json  
1        50.809508    000N144HH73_MC_F_PP_optMW6_10.json  
2        24.198928  000N144HH73_MC_I_MPS_optBD24_10.json  
3        50.991603    000N144HH73_MC_I_PP_optMW6_10.json  
4        47.329537   000N144HH73_MC_TQA_PP_optMW6_5.json  


In [5]:
# Compute approximation ratio for each object and append to the DataFrame
minmax_cache = {} # Since the same instance is present in multiple rows of the DataFrame, just load and store it once
approximation_ratio = []

for _, row in df_hardware.iterrows():
    instance_name = row["instance_name"]
    energy = row["expected_energy"] 
    instance = instance_name[:3]
    if instance not in minmax_cache:
        minmax_instance_path = get_minmax(minmax_path, graph_type, instance, num_nodes) # Extract specific minmax_instance path
        minmax_cache[instance] = extract_minmax_args(minmax_instance_path) # Cache args for the speciic instance
    min_cut, max_cut, sum_weights = minmax_cache[instance] # Extract minmax args for the specific instance
    approximation_ratio.append(maxcut_approximation_ratio(min_cut, max_cut, sum_weights, energy)) # Evlaute approximation ratio for that row
df_hardware["approximation_ratio"] = approximation_ratio

print(df_hardware.head())
    

  instance_name  QPU_time (s)  num_shots problem_class training_method  \
0   000N144HH73             8       4096        maxcut        F_MPS_10   
1   000N144HH73             8       4096        maxcut         F_PP_10   
2   000N144HH73             8       4096        maxcut        I_MPS_10   
3   000N144HH73             8       4096        maxcut         I_PP_10   
4   000N144HH73             8       4096        maxcut    TQA_PP_opt_5   

   expected_energy                           result_file  approximation_ratio  
0        27.752374  000N144HH73_MC_F_MPS_optBD24_10.json             0.722389  
1        50.809508    000N144HH73_MC_F_PP_optMW6_10.json             0.907153  
2        24.198928  000N144HH73_MC_I_MPS_optBD24_10.json             0.693914  
3        50.991603    000N144HH73_MC_I_PP_optMW6_10.json             0.908612  
4        47.329537   000N144HH73_MC_TQA_PP_optMW6_5.json             0.879267  


#### Data Analysis for IBM Quantum QAOA Parameter Training Data ####

In [6]:
instance_paths_training = []

for p in p_list:
    for instance in instance_list:
        instance_paths_training.extend(QAOATraining.locate_training_instance(training_data_path, graph_type, instance, 
        num_nodes, p, ER_probability=None, swap_layers=None, degree=None))

print(instance_paths_training)
print(len(instance_paths_training))

[PosixPath('/mnt/c/Users/rames102/Desktop/QAOA-Parameter-Setting/data/training/heavy_hex/20251020_232931_000N144HH73_MC_TQA_PP_optMW6_5.json'), PosixPath('/mnt/c/Users/rames102/Desktop/QAOA-Parameter-Setting/data/training/heavy_hex/20251021_225719_000N144HH73_MC_TQA_MPS_optBD24_5.json'), PosixPath('/mnt/c/Users/rames102/Desktop/QAOA-Parameter-Setting/data/training/heavy_hex/20260127_092702_000N144HH73_MC_LR_PP_optMW6_5.json'), PosixPath('/mnt/c/Users/rames102/Desktop/QAOA-Parameter-Setting/data/training/heavy_hex/20260129_083149_000N144HH73_MC_LR_MPSAer_opt_DB24_5.json'), PosixPath('/mnt/c/Users/rames102/Desktop/QAOA-Parameter-Setting/data/training/heavy_hex/20251020_224658_001N144HH73_MC_TQA_PP_optMW6_5.json'), PosixPath('/mnt/c/Users/rames102/Desktop/QAOA-Parameter-Setting/data/training/heavy_hex/20251021_225732_001N144HH73_MC_TQA_MPS_optBD24_5.json'), PosixPath('/mnt/c/Users/rames102/Desktop/QAOA-Parameter-Setting/data/training/heavy_hex/20260127_093448_001N144HH73_MC_LR_PP_optMW6_5

In [7]:
all_instances_data_training = []

for instance_path in instance_paths_training:
    obj = QAOATraining.load_training_instance(instance_path)
    all_instances_data_training.append(obj)

# Expand each object into data columns
all_data_dicts_training = []

for obj in all_instances_data_training:
    data_dict_training = {
        "instance_name": obj.instance_name,
        "pre_processing_time": obj.pre_processing_time,
        "pre_processor_name": obj.pre_processor_name,
        "train_duration_per_iter": obj.train_duration_per_iter,
        "expected_energy_per_iter": obj.expected_energy_per_iter,
        "trainer_name_per_iter": obj.trainer_name_per_iter,
        "train_duration_per_layer": obj.train_duration_per_layer,
        "expected_energy_per_layer": obj.expected_energy_per_layer,
        "trainer_name_per_layer": obj.trainer_name_per_layer,
        "num_depth_iter": obj.num_depth_iter,
    }
    all_data_dicts_training.append(data_dict_training)

In [8]:
# Creata a DataFrame to document the extracted training data  
df_training = pd.DataFrame(all_data_dicts_training)
# print(df_training.loc[74, :]["trainer_name_per_iter"])
print(df_training.head())

                              instance_name  pre_processing_time  \
0       000N144HH73_MC_TQA_PP_optMW6_5.json                  NaN   
1     000N144HH73_MC_TQA_MPS_optBD24_5.json           289.975533   
2        000N144HH73_MC_LR_PP_optMW6_5.json           346.529271   
3  000N144HH73_MC_LR_MPSAer_opt_DB24_5.json           407.179045   
4       001N144HH73_MC_TQA_PP_optMW6_5.json                  NaN   

  pre_processor_name                  train_duration_per_iter  \
0               None  [51.947216510772705, 317.8320360183716]   
1          SATMapper  [168.91438794136047, 429.0275182723999]   
2          SATMapper                      [79.94285893440247]   
3          SATMapper                      [928.9998672008514]   
4               None   [45.84358072280884, 234.5539755821228]   

                  expected_energy_per_iter  \
0  [41.77672806116267, 47.329536787543674]   
1  [23.959557152217407, 27.55875064098228]   
2                      [41.77546875464326]   
3               